# Zero-Train Optimization & Maintenance (ZTOM) of a Legal Understanding LLM

**Legal Understanding LLM — [Google Gemma 3 1B IT](https://huggingface.co/google/gemma-3-1b-it) + LoRA (PEFT)**

## Model setup

- **Base model:** [Gemma 3 1B IT](https://huggingface.co/google/gemma-3-1b-it) — (instruction-tuned)
- **Adaptation:** LoRA adapters via [Hugging Face PEFT](https://huggingface.co/docs/peft)
- **Rationale:** Efficient training, standard practice, and straightforward comparison across checkpoints

## Fine-tuning tasks

### Task A — Legal reasoning

Use datasets that already pair inputs with **Yes/No** (or binary) answers where applicable:
- **[nguha/legalbench](https://huggingface.co/datasets/nguha/legalbench)** — LegalBench: many tasks over legal text; see the [dataset card](https://huggingface.co/datasets/nguha/legalbench) and [project site](https://hazyresearch.stanford.edu/legalbench/).
- **[chenghao/cuad_qa](https://huggingface.co/datasets/chenghao/cuad_qa)** — CUAD as QA over contracts; [CUAD paper](https://arxiv.org/abs/2103.06268).

### Task B — Summarization

- **Dataset:** [BillSum](https://huggingface.co/datasets/billsum) — long bills with human summaries.
- **I/O:** long bill or document text → concise summary.
- **Evaluation:** **ROUGE** (e.g. ROUGE-1 / ROUGE-L) vs. reference summaries.

## Scenario

After initial training, the model may benefit from optimization to improve its response quality and accuracy. Traditional approaches would require retraining the model with additional data, which is time-consuming and computationally expensive. ZTOM provides an alternative solution that optimizes the model's performance using only a small validation set and semantic similarity metrics to guide the optimization process.

## Use Case

This demonstration is applicable when:
- A model is already trained and deployed, but performance improvements are desired
- Retraining is not feasible due to time, computational, or data constraints
- You have access to only a small validation set for optimization
- You want to improve response quality using semantic similarity as a guide

## Summary

This notebook shows how **Zero-Train Optimization & Maintenance (ZTOM)** adjusts the Legal Understanding LLM (Gemma 3 1B IT + LoRA) **without retraining**. The objective is **mean string similarity** between model outputs and reference answers (RapidFuzz `partial_ratio`, higher is better), with **`minimize=False`** so ZTOM **maximizes** that score. In this run, the score rose from **~31.2** to **~59.7** (**about +91% vs. the starting value**, or **~1.9×** the baseline), using **100** optimizer evaluations on the validation slice used in the notebook.
  
## ZTOM Result Outputs

| Metric | Value |
|--------|------:|
| **Objective** | Mean RapidFuzz `partial_ratio` vs. `dataset["answer"]` (higher is better) |
| **Optimizer** | `minimize=False` (maximize objective) |
| **Scaling factors** | `[-0.262, 0.084, 0.999, -0.614, -0.226, 0.765, -0.215, 0.577, 0.276]` |
| **Original objective** | 31.227872848510742 |
| **Best objective** | 59.72260665893555 |
| **Relative improvement** | ~91% vs. original \((\text{best}-\text{original})/\text{original}\) |
| **Number of evaluations** | 100 |

### Install dependencies and utility functions: This cell should be run once.

In [1]:
%%capture
%pip install -r requirements.txt
%pip install 'authentrics==0.21.2' --extra-index-url='https://us-central1-python.pkg.dev/authentrics/authentrics/simple'

In [2]:
from authentrics import AuthentricsSession
from dotenv import load_dotenv
from tokenizers import Tokenizer
import pathlib as Path


load_dotenv()

def cleanup_project(session: AuthentricsSession, project_name: str):
    print("Cleaning up project: ", project_name)
    if Path(project_name).exists():
        try:
            projects = session.get_projects()
            print("Projects: found")
            for project in projects:
                if project.name == project_name:
                    session.delete_project(project)
                    print("Project: deleted")
                    Path(project_name).rmdir()
                    print("Project: removed")
                    return
        except Exception as e:
            print("Error: ", e)
            Path(project_name).rmdir()


def format_row(tokenizer: Tokenizer, example):
    messages = [
        {
            "role": "user",
            "content": (
                "Answer the legal question based on the contract.\n\n"
                f"Question: {example['question']}\n\n"
                f"Context:\n{example['context']}"
            ),
        },
        {"role": "assistant", "content": example["answer"]},
    ]
    example["formatted_chat"] = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )
    return example

## Download Checkpoints and Dataset

In [3]:
from pathlib import Path

checkpoint_dir = Path('checkpoints')
data_dir = Path('data')

checkpoint_dir.mkdir(parents=True, exist_ok=True)
data_dir.mkdir(parents=True, exist_ok=True)

!unlink checkpoints/cuad_checkpoints
!unlink checkpoints/bill_sum_v2
!unlink data/test_cuad_qa.json
!ln -s /tmp/draft-webinar-demo/checkpoints/cuad_checkpoints checkpoints/cuad_checkpoints
!ln -s /tmp/draft-webinar-demo/checkpoints/bill_sum_v2 checkpoints/bill_sum_v2
!ln -s /tmp/draft-webinar-demo/datasets/test_cuad_qa.json data/test_cuad_qa.json

checkpoints = [checkpoint_dir / "cuad_checkpoints" / f"checkpoint-{i}" for i in range(300, 1201, 300)]
checkpoints.append(checkpoint_dir / "cuad_checkpoints" / "final_model" / "checkpoint_20260311_231438")
checkpoints += [checkpoint_dir / "bill_sum_v2" / f"checkpoint-{i}" for i in range(300, 1201, 300)]
checkpoints.append(checkpoint_dir / "bill_sum_v2" / "final_model" / "checkpoint_20260315_180509")

data_file = data_dir / "test_cuad_qa.json"
print(checkpoints)

[PosixPath('checkpoints/cuad_checkpoints/checkpoint-300'), PosixPath('checkpoints/cuad_checkpoints/checkpoint-600'), PosixPath('checkpoints/cuad_checkpoints/checkpoint-900'), PosixPath('checkpoints/cuad_checkpoints/checkpoint-1200'), PosixPath('checkpoints/cuad_checkpoints/final_model/checkpoint_20260311_231438'), PosixPath('checkpoints/bill_sum_v2/checkpoint-300'), PosixPath('checkpoints/bill_sum_v2/checkpoint-600'), PosixPath('checkpoints/bill_sum_v2/checkpoint-900'), PosixPath('checkpoints/bill_sum_v2/checkpoint-1200'), PosixPath('checkpoints/bill_sum_v2/final_model/checkpoint_20260315_180509')]


## Authentrics Python Library

In [4]:
from authentrics import AuthentricsException, AuthentricsSession, ZtomOptimizationOptions
from datetime import datetime

PROJECT_NAME = "ZTOM_Local_Sean"
project_path = Path(PROJECT_NAME)

session = AuthentricsSession()
session.login()

try:
    project = session.load_project(project_path)

except AuthentricsException:
    cleanup_project(session, PROJECT_NAME)
    project = session.create_project(project_path, PROJECT_NAME)
    print(checkpoints[-1])
    project = session.add_checkpoints(project, *checkpoints)


Cleaning up project:  ZTOM_Local_Sean
[DEBUG] Status code: 200
checkpoints/bill_sum_v2/final_model/checkpoint_20260315_180509
[INFO] Logged in successfully
[DEBUG] Making POST request to /project
[DEBUG] Body: {"description":"","format":"ONNX","name":"ZTOM_Local_Sean","type":"CLIENT_MANAGED"}
[DEBUG] Status code: 200
[DEBUG] Creating checkpoint for project 69c6c5e6690c8e290bed8793 with filename checkpoint-300 and hash 58271e6bd4d0f6e1a1bd906b6bff5b0f1800e6766e15156026601f623b24fcc1
[DEBUG] Making POST request to /project/file/external
[DEBUG] Body: {"fileName":"checkpoint-300","filePath":"LOCAL","format":"ONNX","hash":"58271e6bd4d0f6e1a1bd906b6bff5b0f1800e6766e15156026601f623b24fcc1","projectId":"69c6c5e6690c8e290bed8793"}
[DEBUG] Status code: 200
[DEBUG] Successfully created checkpoint: {"id":"69c6c5e6690c8e290bed8793","name":"ZTOM_Local_Sean","description":"","format":"ONNX","createdOn":1774634470906,"modifiedOn":null,"baseModel":null,"type":"CLIENT_MANAGED","fileList":[{"id":"69c6c5

In [5]:
project

Project(id=69c6c5e6690c8e290bed8793, name='ZTOM_Local_Sean', description='', created_at='Fri Mar 27 18:01:10.906000000 2026', project_path='optional("/home/seanhagstrom/core/demos/ZTOM_Local/ZTOM_Local_Sean")')

In [6]:
project.checkpoints

[Checkpoint(id=69c6c5e9690c8e290bed8795, name='checkpoint-300', hash=58271e6bd4d0f6e1a1bd906b6bff5b0f1800e6766e15156026601f623b24fcc1, created_at='Fri Mar 27 18:01:13.186000000 2026', path='optional("/tmp/draft-webinar-demo/checkpoints/cuad_checkpoints/checkpoint-300")'),
 Checkpoint(id=69c6c5eb690c8e290bed8796, name='checkpoint-600', hash=6555d5a9df97f95368fbee731a797a3f74e1e81288eba44718e59a69af4359e6, created_at='Fri Mar 27 18:01:15.424000000 2026', path='optional("/tmp/draft-webinar-demo/checkpoints/cuad_checkpoints/checkpoint-600")'),
 Checkpoint(id=69c6c5ed690c8e290bed8797, name='checkpoint-900', hash=ba95fa6a4b5892f36f94b554cd2a7f3b110f1b8ae4c7c21903ba2929aafa6ada, created_at='Fri Mar 27 18:01:17.516000000 2026', path='optional("/tmp/draft-webinar-demo/checkpoints/cuad_checkpoints/checkpoint-900")'),
 Checkpoint(id=69c6c5ef690c8e290bed8798, name='checkpoint-1200', hash=3a1a5d5dd8415d656acfc18675a850f50def3185135245c6ed46d9b07c8ad1b7, created_at='Fri Mar 27 18:01:19.647000000 202

### Model Wrapper

In [7]:
from authentrics import InferenceResult, ModelInterface, Parameters, WeightBias, use_backend
from datasets import Column, Dataset
from transformers import TextGenerationPipeline
from transformers.pipelines import pipeline


class SimpleHFModel(ModelInterface):
    def __init__(self, dataset: Dataset | None = None, batch_size: int = 1):
        super().__init__()

        use_backend("torch")

        self._dataset = dataset or Dataset.from_list([])
        self._batch_size = batch_size

        self._inference_config = {"max_new_tokens": 50, "do_sample": False, "return_full_text": False}

        self._module = None
        self._input_data = None

    def load(self, checkpoint_path: Path | str | bytes) -> None:
        self._module: TextGenerationPipeline = pipeline(
            "text-generation",
            model=str(checkpoint_path),
            trust_remote_code=True,
            # device_map="sequential",
        )

        if self._input_data is None:
            formatted_chat: Column = self._dataset.map(
                lambda example: format_row(self._module.tokenizer, example)
            )["formatted_chat"]
            self._input_data = list(formatted_chat)

    def get_weight_bias(
        self,
        weight_names: list[str] | None = None,
        bias_names: list[str] | None = None,
    ) -> WeightBias:
        weights = Parameters()
        biases = Parameters()
        for name, param in self._module.model.named_parameters():
            last_part = name.rsplit(".", 1)[-1]
            if last_part == "weight":
                if weight_names is None or name in weight_names:
                    weights[name] = param.detach().cpu()

            elif last_part == "bias":
                if bias_names is None or name in bias_names:
                    biases[name] = param.detach().cpu()

        return WeightBias(weights, biases)

    def perform_inference(
        self,
        return_intermediate_outputs: bool = False,
        layer_names: list[str] | None = None,
    ) -> InferenceResult:
        chat_template = self._inference_config.pop("chat_template", None)

        assert self._module.tokenizer is not None
        if chat_template is not None:
            self._module.tokenizer.chat_template = chat_template

        result = self._module(
            text_inputs=self._input_data,
            batch_size=self._batch_size,
            chat_template=chat_template,
            **self._inference_config,
        )

        return InferenceResult([r[0]["generated_text"] for r in result])

    def set_weight_bias(self, weight_bias: WeightBias) -> None:
        for name, tensor in self._module.model.named_parameters():
            if name in weight_bias.weights:
                tensor.data.copy_(weight_bias.weights[name])
            if name in weight_bias.biases:
                tensor.data.copy_(weight_bias.biases[name])

    def save(self, checkpoint_path: Path | str | bytes) -> None:
        path = Path(checkpoint_path)
        path.parent.mkdir(parents=True, exist_ok=True)

        # Avoid a bug in the Hugging Face pipeline
        if not hasattr(self._module, "modelcard"):
            self._module.modelcard = None

        self._module.save_pretrained(path)


### Prepare the Data

In [8]:
from datasets import load_dataset


dataset = load_dataset(
    "json",
    data_files={"eval": str(data_file)},
)["eval"].take(100)

### Run ZTOM Model Maintenance

#### Setup loss function

This is a custom loss function that calculates the average fuzzy match between the predicted and actual answers. We will maximize this function over the course of the optimization process.

In [9]:
import torch
from rapidfuzz import fuzz


def clean_answer(text: str) -> str:
    if "<start_of_turn>model" in text:
        text = text.split("<start_of_turn>model")[-1]

    return text.strip()


def average_fuzzy_match(y_pred: list[str], answers: list[str]) -> float:
    if len(y_pred) != len(answers):
        raise ValueError("y_pred and answers must have the same length")

    total = len(y_pred)
    if total == 0:
        return 0.0

    scores = torch.tensor(
        [
            fuzz.partial_ratio(clean_answer(output).lower(), answer.lower())
            for output, answer in zip(y_pred, answers)
        ]
    )

    return scores.mean().item()


In [10]:
from transformers import logging


logging.set_verbosity(logging.ERROR)

session.model = SimpleHFModel(dataset=dataset, batch_size=10)

options = ZtomOptimizationOptions(
    max_evaluations=100,
    xtol_rel=1e-4,
    ftol_rel=1e-4,
    lower_bound=-1.0,
    upper_bound=1.0,
    minimize=False,
)

ztom_result = session.ztom_analysis(
    project,
    lambda y_pred: average_fuzzy_match(y_pred, dataset["answer"]),
    project_path / "optimized_checkpoint",
    options,
)

print(ztom_result)

[INFO] Registered tensor factory for backend: torch
[DEBUG] Making GET request to /api/auth/user
[DEBUG] Status code: 200
[DEBUG] Making POST request to /api/auth/user/permission
[DEBUG] Body: {"endpoint":"/zto/batch","projectId":"69c6c5e6690c8e290bed8793"}
[DEBUG] Status code: 200
[INFO] Permission Granted for user:, endpoint:/zto/batch, grantedBy: ProjectAuthorizationManager, grantedAt: 1774634503044, projectId:69c6c5e6690c8e290bed8793
[DEBUG] Making GET request to /project/69c6c5e6690c8e290bed8793
[DEBUG] Status code: 200


Optimized checkpoint path: ZTOM_Local_Sean/optimized_checkpoint, Scaling factors: [-0.262144963205403, 0.08410686914166096, 0.9999999999999999, -0.6143650695961679, -0.22678029601024013, 0.7652433657298816, -0.21543835940823433, 0.5774992561087838, 0.27685830399828126], Original loss: 31.227872848510742, Best loss: 59.72260665893555, Number of inferences: 100
